In [ ]:
import pathlib
import pickle
from distutils.command.bdist import bdist
from adaptive_latents.regressions import BaseKernelRegressor

import adaptive_latents.stim_designer

from adaptive_latents import StimRegressor
from adaptive_latents.stim_designer import StimDesigner
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import importlib


rng = np.random.default_rng(0)

In [ ]:
def proportion_in_space(desired, designed):
    assert np.allclose(desired.T @ desired, np.eye(desired.shape[1]))
    proj = desired @ desired.T @ designed
    in_norm = np.linalg.norm(proj)
    total_norm = np.linalg.norm(designed)
    if total_norm == 0:
        ratio = 0
    else:
        ratio = in_norm / total_norm
    return ratio


In [ ]:
with open(pathlib.Path('/home/jgould/Documents/neurips_2025/generated/') / 'optimization_history.pkl', 'rb') as fhan:
    sr = pickle.load(fhan)
sr: StimRegressor

In [ ]:
sr.stim_designer.log[-1].keys()

In [ ]:
l = sr.stim_designer.log[39]
fig, axs = plt.subplots(nrows=2, figsize=(10, 5), height_ratios=[1,1])
axs[0].plot(l['v'],'.-', label='v (target)')
axs[0].plot(l['s'], '.-', label='s (effect predicted in optimization)')
axs[0].plot(l['observed_s_hat'], label='corresponding $\hat s$ entry')
axs[0].legend()

# axs[1].plot(l['observed_s_hat'].T, label='$\hat s$ (observed effect)')
u = l['u']
axs[1].plot(u, label=f'u (high-d stim) ($L_0={np.linalg.norm(u, ord=0).astype(int)}$)')
axs[1].legend()

In [ ]:
u_l0s = []
ratios = []
latents = []
s_errors = []
selected_v = []
sr_preq_eval = []


last_stim_reg = None
for i, l in enumerate(sr.stim_designer.log):
# for l in sd.log:
    s = l['s']
    u = l['u']
    v = l['v']
    latent = np.argmax(v.flatten())

    ratios.append(proportion_in_space(v, s))
    u_l0s.append(np.linalg.norm(u, ord=0))
    latents.append(latent)
    if 'observed_s_hat' in l:
        observed_s_hat = l['observed_s_hat']
        stim_reg = l['stim_reg']
        s_errors.append(np.sqrt(np.mean((observed_s_hat - s)**2)))

        if last_stim_reg is not None:
            error = stim_reg.history[i-1,stim_reg.input_d:] - last_stim_reg.predict(stim_reg.history[i-1, :stim_reg.input_d])
            sr_preq_eval.append(np.linalg.norm(error))
        else:
            sr_preq_eval.append(np.nan)

        last_stim_reg = stim_reg

    selected_v.append(np.argmax(np.abs(v.flatten())))

n_rows = 5
fig, axs = plt.subplots(nrows=n_rows, squeeze=False, layout='constrained', figsize=(10, n_rows*2))
# ax.plot(u_l0s, '.')
ax = axs[0,0]
ax.plot(ratios, '.-')
ax.set_title('proportion of s along v')

ax = axs[1,0]
axs[1,0].plot(u_l0s, '.-')
axs[1,0].axhline(30, color='r')
axs[1,0].set_title('L0 norm of u')

ax = axs[2,0]
ax.plot(s_errors, '.-')
ax.set_title('s prediction rmse')

ax = axs[3,0]
ax.plot(selected_v, '.-')
ax.set_title('index of biggest-magnitude v entry')

ax = axs[4,0]
ax.plot(sr_preq_eval, '.-')
ax.set_title(' $\hat S$ prequential error')



In [ ]:

plt.scatter(latents,ratios,s=10)
plt.xlabel('latent chosen to stimulate')
plt.ylabel('proportion of s along v')


In [ ]:
a_s = np.linspace(-8, 2.5, 1) # 20
a_s_label = 'log L1 coeff'
b_s = np.linspace(30, 40, 1).astype(int) # 4
b_s_label = 'max L0 norm (target)'

depth = 1


accumulators = {'time':[], 'L0':[], 'low-d in-prop':[], 'high-d in-prop':[], 'sd': [], 'low-d in-prop (u.t.)': []}
times_accumulator = []
objective_accumulator = []

with tqdm(total=len(a_s) * len(b_s) * depth * len(sr.stim_designer.log)) as pbar:
    for a in a_s:
        [acc.append([]) for acc in accumulators.values()]
        for b in b_s:
            [acc[-1].append([]) for acc in accumulators.values()]
            for _ in range(depth):
                [acc[-1][-1].append([]) for acc in accumulators.values()]
                sd = StimDesigner(
                    max_l0_norm=b,
                    rng_seed=rng.integers(2**32),
                    should_log=True,
                )


                for l in sr.stim_designer.log:
                    v = l['v']
                    pro = l['pro']

                    u_to_s_function=lambda u: pro.Q.T @ u * .0001
                    s_to_u_function=lambda s: pro.Q @ s
                    new_u, designed_s = sd.design_stim(v, u_dimension=pro.Q.shape[0], u_to_s_function=u_to_s_function)


                    accumulators['high-d in-prop'][-1][-1][-1].append(proportion_in_space(s_to_u_function(v), new_u))
                    accumulators['low-d in-prop'][-1][-1][-1].append(proportion_in_space(v, designed_s))
                    accumulators['low-d in-prop (u.t.)'][-1][-1][-1].append(proportion_in_space(v, sd.log[-1]['unthresholded_s']))
                    accumulators['time'][-1][-1][-1].append(sd.log[-1]['time'] * 1000)
                    accumulators['L0'][-1][-1][-1].append(np.linalg.norm(new_u, ord=0))
                    accumulators['sd'][-1][-1][-1].append(sd)
                    pbar.update(1)

accumulators = {k:np.array(v) for k,v in accumulators.items()}
# accumulators['in/out ratio'][np.isnan(accumulators['in/out ratio'])] = np.nanmax(accumulators['in/out ratio'])


shading = 'nearest'
if len(a_s) == 1 or len(b_s) == 1:
    shading = 'flat'

def grid_f(b_s, shading):
    if shading == 'nearest':
        return b_s
    elif shading == 'flat':
        if len(b_s) == 1:
            return [b_s[0]-1e-12, b_s[0]+1e-12]
        else:
            return np.hstack([b_s, b_s[-1] + b_s[-1] - b_s[-2]])

as_array, bs_array = np.meshgrid(grid_f(b_s, shading),grid_f(a_s, shading))


In [ ]:
fig, axs = plt.subplots(squeeze=False, ncols=2, nrows=3, figsize=(10, 15), layout='constrained', sharex=True, sharey=True,)

ax = axs[0,0]
im = ax.pcolormesh(bs_array, as_array, np.median(accumulators['time'], axis=(-2, -1)), shading=shading)
c = fig.colorbar(im)
ax.set_xticks(a_s)
ax.set_xticklabels(labels=[f'{a:.3f}' for a in a_s], rotation=45)
ax.set_yticks(b_s)
ax.set_title('max opt. time over all calls in s-s run (ms)')

ax = axs[0,1]
cutoff = 30
im = ax.pcolormesh(bs_array, as_array, np.mean(accumulators['L0'] <= cutoff, axis=(-2, -1)), shading=shading, vmax=1)
c = fig.colorbar(im)
ax.set_xticks(a_s)
ax.set_xticklabels(labels=[f'{a:.3f}' for a in a_s], rotation=45)
ax.set_yticks(b_s)
ax.set_title(f'proportion of stimuli with L0<={cutoff}')

ax = axs[1,0]
im = ax.pcolormesh(bs_array, as_array, np.min(accumulators['high-d in-prop'], axis=(-2, -1)), shading=shading, vmax=1)
c = fig.colorbar(im)
ax.set_xticks(a_s)
ax.set_xticklabels(labels=[f'{a:.3f}' for a in a_s], rotation=45)
ax.set_yticks(b_s)
ax.set_title(f'min high-d in-prop')

ax = axs[1,1]
im = ax.pcolormesh(bs_array, as_array, np.min(accumulators['low-d in-prop'], axis=(-2, -1)), shading=shading, vmax=1)
c = fig.colorbar(im)
ax.set_xticks(a_s)
ax.set_xticklabels(labels=[f'{a:.3f}' for a in a_s], rotation=45)
ax.set_yticks(b_s)
ax.set_title(f'min low-d in-prop')

ax = axs[2,0]
difference = (accumulators['low-d in-prop'] == accumulators['low-d in-prop (u.t.)']).mean(axis=(-2, -1))
im = ax.pcolormesh(bs_array, as_array, difference, shading=shading)
c = fig.colorbar(im)
ax.set_xticks(a_s)
ax.set_xticklabels(labels=[f'{a:.3f}' for a in a_s], rotation=45)
ax.set_yticks(b_s)
ax.set_title(f"proportion of trials that didn't need threshold" )


ax = axs[2,1]
im = ax.pcolormesh(bs_array, as_array, np.mean(accumulators['L0'] == 0, axis=(-2, -1)), shading=shading)
c = fig.colorbar(im)
ax.set_xticks(a_s)
ax.set_xticklabels(labels=[f'{a:.3f}' for a in a_s], rotation=45)
ax.set_yticks(b_s)
ax.set_xlabel(f'a_s: {a_s_label}')
ax.set_title(f'proportion of all 0 returns')


for ax in axs[:,0]:
    ax.set_ylabel(f'b_s: {b_s_label}')
    ax.set_xlabel(f'a_s: {a_s_label}')

for ax in axs[-1]:
    ax.set_ylabel(f'b_s: {b_s_label}')
    ax.set_xlabel(f'a_s: {a_s_label}')



In [ ]:
s = ( np.argmin(np.abs(a_s - -5.1)), np.nonzero(30==b_s)[0], 0, slice(None, None))
fig, ax = plt.subplots()
ax.hist(accumulators['high-d in-prop'][s].flatten());
ax.set_xlabel('high-d in-prop')
ax.set_ylabel("# of $u$'s")



In [ ]:
importlib.reload(adaptive_latents.stim_designer)

sd = adaptive_latents.stim_designer.StimDesigner(
    max_l0_norm=30,
    # adaptive_starter_lam_1=False,
    starter_lam_1_guess=1.2,
    max_outer_loop_time_ms=500, # or 20
    # max_inner_iters=60, # or 60
    convergence_threshold=1e-3,
    rng_seed=1,
    should_log=True,
    # a1=a
)

for l in sr.stim_designer.log:
    v = l['v']
    pro = l['pro']

    new_u = sd.design_stim(v, u_dimension=pro.Q.shape[0], u_to_s_function=lambda u: pro.Q.T @ u)



In [ ]:
l = sd.log[6]
    # 'time', 'v', 's', 'loss_history', 'lam_1_history', 'l0_history', 's_history'
lam_1_history = l['lam_1_history']
loss_history = l['loss_history']
s = l['u']
s_history = l['s_history']

print(np.linalg.norm(s_history[-1][-1],ord=0))


flat_loss_history = np.squeeze(np.hstack(loss_history))
flat_lam_1_history = np.hstack([[lam_1] *len(loss_h) for lam_1, loss_h  in zip(lam_1_history, loss_history)])
flat_s_history = np.vstack(s_history).T
optimal_index = np.argmax(((flat_s_history / flat_s_history.max(axis=0)).T == s).all(axis=1))
fig, axs = plt.subplots(nrows=3, sharex=True, figsize=(5, 10), height_ratios=[1,1,2])
axs[0].plot(flat_loss_history)
axs[0].set_title('loss over time')
axs[1].plot(np.log(flat_lam_1_history))
axs[1].set_title('L1 coefficient over time')

axs[2].imshow(flat_s_history, aspect='auto', interpolation='none')
axs[2].set_title('u over time')

axis = axs[2].axis()
axs[2].set_xticks(list(axs[2].get_xticks()) + [optimal_index], labels=list(axs[2].get_xticklabels()) + [f'\n{optimal_index}'])
axs[2].axis(axis)

In [ ]:
plt.plot(l['predicted_s'])

In [ ]:
def plot_over_all_history(log):
    all_loss_history  = []
    all_lam_1_history  = []
    all_s_history  = []

    for l in log:
        # 'time', 'v', 's', 'loss_history', 'lam_1_history', 'l0_history', 's_history'
        lam_1_history = l['lam_1_history']
        loss_history = l['loss_history']
        u = l['u']
        s_history = l['s_history']

        flat_loss_history = np.squeeze(np.hstack(loss_history))
        flat_lam_1_history = np.hstack([[lam_1] *len(loss_h) for lam_1, loss_h  in zip(lam_1_history, loss_history)])
        flat_s_history = np.vstack(s_history).T
        optimal_index = np.argmax(((flat_s_history / flat_s_history.max(axis=0)).T == u).all(axis=1))

        all_loss_history.append(flat_loss_history)
        all_lam_1_history.append(flat_lam_1_history)
        all_s_history.append(flat_s_history)





    fig, axs = plt.subplots(nrows=3, ncols=2, sharex=True, figsize=(10, 10), height_ratios=[1,1,2], layout='constrained')
    axs[0,0].plot(np.hstack(all_loss_history))
    axs[0,0].set_title('loss over time')

    axs[1,0].plot(np.log(np.hstack(all_lam_1_history)))
    axs[1,0].set_title('L1 coefficient over time')

    all_s_history = np.hstack(all_s_history)
    axs[2,0].imshow(all_s_history, aspect='auto', interpolation='none')
    axs[2,0].set_title('u over time')

    axs[2,1].imshow(all_s_history / all_s_history.max(axis=1)[:,None], aspect='auto', interpolation='none')
    axs[2,1].set_title('u normalized by max over time')

    axs[0,1].plot(np.linalg.norm(all_s_history,axis=0, ord=0))
    axs[0,1].axhline(30,color='r')
    axs[0,1].set_title('L0 norm over time')

    axs[1,1].set_title('in/out ratio over time')
    #
    # axis = axs[2].axis()
    # axs[2].set_xticks(list(axs[2].get_xticks()) + [optimal_index], labels=list(axs[2].get_xticklabels()) + [f'\n{optimal_index}'])
    # axs[2].axis(axis)

plot_over_all_history(sr.stim_designer.log)
# plot_over_all_history(sr.log)



In [ ]:
importlib.reload(adaptive_latents.stim_designer)

new_sd =  adaptive_latents.stim_designer.StimDesigner(max_l0_norm=20, should_log=True)

l = sr.stim_designer.log[0]
v = l['v']
pro = l['pro']


u = new_sd.design_stim(v, u_dimension=pro.Q.shape[0], u_to_s_function=lambda u: pro.Q.T @ u)


print(new_sd.log[0].keys())
print(new_sd.log[0]['loss_history'])
plt.plot(np.array(new_sd.log[0]['loss_history']).T)
print(np.array(new_sd.log[0]['l0_history']))

print(new_sd.log[0]['s_history'][0][0])
print(new_sd.log[0]['lam_1_history'][-1])
print(u)
print(np.linalg.norm(u))
print(np.count_nonzero(u))
print(np.argwhere(u>0).T)
